# Kodex — treinamento do zero

Este notebook treina **um modelo experimental com pesos inicializados aleatoriamente**. Ele não carrega pesos de Codex, Copilot, GPT ou outro modelo pré-treinado.

## Objetivo desta etapa
1. Preparar um corpus com origem/licença declaradas.
2. Treinar um tokenizer BPE do próprio corpus.
3. Treinar um pequeno Transformer causal do zero.
4. Salvar o checkpoint e o tokenizer no Google Drive.
5. Gerar uma amostra sem executar o código produzido.

**Limite:** o corpus de demonstração é pequeno demais para criar uma IA competente em programação. Use-o para validar a pipeline. A capacidade real depende de corpus maior e legalmente revisado, compute, tempo de treino e avaliações independentes.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/Security-lab")
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", "kodex-programming-ai",
        "https://github.com/japaguimarassi-create/Security-lab.git",
        str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers"], check=True)

import torch
import tokenizers

print("PyTorch:", torch.__version__)
print("Tokenizers:", tokenizers.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU assigned. The demo can use CPU; larger runs will be slow.")

## 1. Preparar o espaço de trabalho

Os arquivos em `MyDrive/Kodex/source-code` e o manifesto `source-manifest.jsonl` são opcionais.

Para usar seu próprio corpus, coloque **somente arquivos que você tem direito e permissão para usar em treinamento** em `MyDrive/Kodex/source-code`. O manifesto deve conter uma entrada JSON por linha e incluir os campos `path`, `license`, `training_permitted`, `source_url` e `commit`. O valor `training_permitted` precisa ser `true`; a licença precisa estar na lista do script. O hash de commit deve identificar a versão de origem.

O filtro bloqueia certos caminhos, extensões, duplicatas e padrões conhecidos de segredos, mas nenhum scanner consegue garantir que todos os segredos sejam encontrados. Verifique os arquivos e o relatório manualmente.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_KODEX = Path("/content/drive/MyDrive/Kodex")
DRIVE_KODEX.mkdir(parents=True, exist_ok=True)

SOURCE_ROOT = DRIVE_KODEX / "source-code"
MANIFEST = DRIVE_KODEX / "source-manifest.jsonl"
CORPUS = DRIVE_KODEX / "approved-corpus.txt"
CORPUS_REPORT = DRIVE_KODEX / "corpus-report.json"
TOKENIZER_PATH = DRIVE_KODEX / "kodex-tokenizer.json"
CHECKPOINT_PATH = DRIVE_KODEX / "kodex-from-scratch.pt"

SMOKE_CORPUS = REPO / "training/data/smoke_corpus.txt"
USING_SMOKE_CORPUS = True

if MANIFEST.is_file() and SOURCE_ROOT.is_dir():
    subprocess.run([
        sys.executable, str(REPO / "training/prepare_corpus.py"),
        "--manifest", str(MANIFEST),
        "--root", str(SOURCE_ROOT),
        "--output", str(CORPUS),
        "--report", str(CORPUS_REPORT)
    ], check=True)
    USING_SMOKE_CORPUS = False
else:
    CORPUS = SMOKE_CORPUS
    print("Using the tiny smoke corpus. It is for pipeline testing, not real capability.")

corpus_bytes = CORPUS.stat().st_size
print("Corpus:", CORPUS)
print("Corpus size:", f"{corpus_bytes:,}", "bytes")
print("Smoke corpus:", USING_SMOKE_CORPUS)

## 2. Treinar o tokenizer BPE do zero

O tokenizer é treinado a partir do texto selecionado, sem carregar um vocabulário pré-treinado. Para o corpus minúsculo, usamos um vocabulário pequeno; para um corpus de código revisado, o tamanho pode ser aumentado. O limite de memória da implementação atual exige começar com arquivos de tamanho moderado: ela carrega o texto/token IDs em memória, não é ainda uma pipeline distribuída de vários terabytes.

In [ ]:
VOCAB_SIZE = 512 if USING_SMOKE_CORPUS else 8192

subprocess.run([
    sys.executable, str(REPO / "training/train_tokenizer.py"),
    "--data", str(CORPUS),
    "--output", str(TOKENIZER_PATH),
    "--vocab-size", str(VOCAB_SIZE)
], check=True)

print("Tokenizer saved:", TOKENIZER_PATH)
print("Tokenizer size:", f"{TOKENIZER_PATH.stat().st_size:,}", "bytes")

## 3. Treinar os pesos aleatórios

A configuração desta etapa é deliberadamente pequena. Mesmo que a perda diminua, isso **não** demonstra que o modelo consegue programar; a validação usa apenas outra parte do mesmo arquivo. Modelos úteis precisam de muito mais tokens, tarefas de avaliação separadas e testes de código.

In [ ]:
if USING_SMOKE_CORPUS:
    STEPS = 100
    BATCH_SIZE = 4
    BLOCK_SIZE = 64
    LAYERS = 2
    HEADS = 2
    EMBEDDING_SIZE = 64
else:
    STEPS = 1000 if torch.cuda.is_available() else 100
    BATCH_SIZE = 4
    BLOCK_SIZE = 256
    LAYERS = 4
    HEADS = 4
    EMBEDDING_SIZE = 256

TRAIN_ARGS = [
    sys.executable, str(REPO / "training/train.py"),
    "--data", str(CORPUS),
    "--tokenizer", str(TOKENIZER_PATH),
    "--output", str(CHECKPOINT_PATH),
    "--steps", str(STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--block-size", str(BLOCK_SIZE),
    "--layers", str(LAYERS),
    "--heads", str(HEADS),
    "--embedding-size", str(EMBEDDING_SIZE),
    "--eval-interval", str(max(1, STEPS // 10)),
    "--device", "auto"
]
subprocess.run(TRAIN_ARGS, check=True)

assert CHECKPOINT_PATH.is_file() and CHECKPOINT_PATH.stat().st_size > 0
print("Checkpoint saved to Google Drive:", CHECKPOINT_PATH)
print("Weights were initialized and trained in this run; no pretrained model weights were loaded.")

## 4. Gerar uma amostra

O texto gerado é apenas uma amostra para inspecionar o pipeline. Este notebook não executa o conteúdo gerado, não dá acesso ao modelo a ferramentas externas e não altera repositórios.

In [ ]:
subprocess.run([
    sys.executable, str(REPO / "training/generate.py"),
    "--checkpoint", str(CHECKPOINT_PATH),
    "--prompt", "def add(a, b):",
    "--max-new-tokens", "80",
    "--top-k", "20",
    "--device", "auto"
], check=True)

## 5. Próximos passos

- Aumentar e auditar o corpus antes de aumentar a rede.
- Separar dados de treinamento, validação e benchmark por repositório/projeto para evitar vazamento.
- Medir loss, tokens vistos, duração, uso de memória e hash do checkpoint.
- Construir testes de capacidade em tarefas não vistas, com compilação e testes reais.
- Treinar instrução/uso de ferramentas apenas depois que o modelo base tiver competência mensurável.
- Implementar e testar sandbox, permissões, aprovação e trilha de auditoria **antes de permitir que o modelo execute comandos ou edite projetos reais**.

O acesso gratuito a GPU no Colab depende de disponibilidade e tem limites variáveis. Esta notebook não garante treinamento de um modelo grande nem substitui a infraestrutura necessária para isso.

Fontes: [Colab FAQ](https://research.google.com/colaboratory/intl/pt-BR/faq.html) · [Treinamento do zero](https://huggingface.co/docs/course/chapter7/6) · [Planejamento de escala](https://arxiv.org/abs/2203.15556)